# Moduł 6: Klasyczne algorytmy ML

## Spis treści
1. Drzewa decyzyjne
2. Random Forest (las losowy)
3. SVM (Support Vector Machine)
4. Grid Search — dobór hiperparametrów
5. Porównanie algorytmów
6. **Zadania**

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris, make_classification, load_wine
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report
sns.set_style("whitegrid")

## 1. Drzewa decyzyjne

**Idea:** Podejmuj serię pytań typu TAK/NIE, aby sklasyfikować dane. Drzewo dzieli przestrzeń cech na prostokątne regiony.

```
 [petal_length ≤ 2.45?]
 / \
 TAK NIE
 Setosa [petal_width ≤ 1.75?]
 / \
 Versicolor Virginica
```

### Kryterium podziału — formalnie

Niech $S$ to zbiór próbek w węźle, a $p_i = \frac{|S_i|}{|S|}$ — frakcja klasy $i$.

**Gini impurity** (domyślne w sklearn):
$$G(S) = 1 - \sum_{i=1}^{C} p_i^2$$

**Entropia (Information Entropy):**
$$H(S) = -\sum_{i=1}^{C} p_i \log_2 p_i$$

Wartości skrajne:
- $G = 0$ lub $H = 0$ → węzeł idealnie czysty (jedna klasa)
- $G = 1 - \frac{1}{C}$ (max) → równomierny rozkład klas

### Information Gain — wybór najlepszego podziału

Dla cechy $A$ dzielącej $S$ na podzbiory $S_1, S_2, \ldots, S_V$:

$$\text{IG}(S, A) = H(S) - \sum_{v=1}^{V} \frac{|S_v|}{|S|} H(S_v)$$

Wybieramy cechę $A^* = \arg\max_A \text{IG}(S, A)$ — tę, która daje **największy spadek** nieczystości.

### Algorytm CART (Classification And Regression Trees)

Dla podziału binarnego w punkcie $t$ cechy $j$:

$$\text{Cost}(j, t) = \frac{n_{\text{left}}}{n} \cdot G(S_{\text{left}}) + \frac{n_{\text{right}}}{n} \cdot G(S_{\text{right}})$$

Szukamy $(j^*, t^*) = \arg\min_{j,t} \text{Cost}(j, t)$

### Drzewa regresyjne

Dla regresji kryterium to **MSE** w węźle:

$$\text{MSE}(S) = \frac{1}{|S|} \sum_{i \in S} (y_i - \bar{y}_S)^2$$

gdzie $\bar{y}_S$ to średnia wartości w węźle.

### Kontrola złożoności (regularyzacja)

| Parametr | Efekt |
|----------|-------|
| `max_depth` | Maksymalna głębokość drzewa |
| `min_samples_split` | Min. próbek do dalszego podziału |
| `min_samples_leaf` | Min. próbek w liściu |
| `max_features` | Liczba cech rozważanych przy podziale |
| `ccp_alpha` | Cost Complexity Pruning: $R_\alpha(T) = R(T) + \alpha \cdot |T|$ |

### Zalety
- Interpretowalność — można wizualizować drzewo
- Nie wymaga standaryzacji
- Obsługuje dane kategoryczne i brakujące wartości

### Wady
- Łatwo overfituje (głębokie drzewa)
- Niestabilne — mała zmiana danych → zupełnie inne drzewo (wysoka wariancja)
- Granice decyzji zawsze prostopadłe do osi (axis-aligned)

In [ ]:
# Iris — drzewo decyzyjne
iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
 iris.data, iris.target, test_size=0.2, random_state=42
)

# max_depth ogranicza głębokość → kontrola overfittingu
dt = DecisionTreeClassifier(max_depth=3, random_state=42)
dt.fit(X_train, y_train)

print(f"Train accuracy: {dt.score(X_train, y_train):.3f}")
print(f"Test accuracy: {dt.score(X_test, y_test):.3f}")

# Wizualizacja drzewa
plt.figure(figsize=(16, 8))
plot_tree(dt, feature_names=iris.feature_names, class_names=iris.target_names,
 filled=True, rounded=True, fontsize=10)
plt.title("Drzewo decyzyjne — Iris")
plt.show()

In [ ]:
# Feature importance — które cechy są najważniejsze?
importance = dt.feature_importances_
features = iris.feature_names

plt.figure(figsize=(8, 4))
plt.barh(features, importance, color="steelblue")
plt.xlabel("Importance")
plt.title("Ważność cech w drzewie decyzyjnym")
plt.show()

## 2. Random Forest (Las Losowy)

**Idea:** Zamiast jednego drzewa, trenujemy **wiele drzew** na losowych podzbiorach danych i **agregujemy** ich predykcje. To zastosowanie techniki **bagging** (Bootstrap Aggregating).

### Bagging — formalnie

1. Z oryginalnego zbioru $\mathcal{D}$ o rozmiarze $n$ tworzymy $B$ zbiorów bootstrapowych $\mathcal{D}_1, \ldots, \mathcal{D}_B$, każdy przez losowanie **ze zwracaniem** $n$ próbek.

2. Trenujemy $B$ niezależnych modeli (drzew) $h_1, \ldots, h_B$.

3. **Agregacja predykcji:**
 - Klasyfikacja (głosowanie większościowe): $\hat{y} = \text{mode}\{h_1(x), h_2(x), \ldots, h_B(x)\}$
 - Regresja (uśrednianie): $\hat{y} = \frac{1}{B}\sum_{b=1}^{B} h_b(x)$

### Dlaczego bagging zmniejsza wariancję?

Jeśli drzewa mają wariancję $\sigma^2$ i korelację $\rho$, to wariancja średniej:

$$\text{Var}\left(\frac{1}{B}\sum_{b=1}^{B} h_b\right) = \rho \sigma^2 + \frac{1 - \rho}{B} \sigma^2$$

Im mniejsza korelacja $\rho$ między drzewami → tym większa redukcja wariancji.

### Random Subspace — dekorelacja drzew

Każdy podział w drzewie rozważa **losowy podzbiór** $m$ cech z $d$ dostępnych:
- Klasyfikacja: $m = \lfloor\sqrt{d}\rfloor$ (domyślnie)
- Regresja: $m = \lfloor d/3 \rfloor$

To **dekoreluje** poszczególne drzewa, zwiększając efektywność ensemblu.

### Out-of-Bag (OOB) Error

Każda próbka nie pojawia się w ~36.8% bootstrapów (prawdopodobieństwo pominięcia: $(1-\frac{1}{n})^n \approx e^{-1} \approx 0.368$).

Te próbki można użyć jako **wbudowany zbiór walidacyjny** — to **OOB error**, estymator błędu bez potrzeby cross-validation.

### Feature Importance (MDI)

Mean Decrease in Impurity dla cechy $j$:

$$\text{Imp}(j) = \sum_{\text{węzły } t \text{ dzielące po } j} \frac{n_t}{n} \cdot \Delta G(t)$$

uśrednione po wszystkich drzewach w lesie.

### Zalety vs drzewo
- Dużo odporniejszy na overfitting (redukcja wariancji)
- Bardziej stabilny
- Jeden z najlepszych algorytmów "out of the box"
- OOB error jako wbudowana walidacja

### Gradient Boosting — alternatywa

W przeciwieństwie do baggingu, **boosting** buduje drzewa **sekwencyjnie**, gdzie każde kolejne drzewo koryguje błędy poprzednich:

$$F_m(x) = F_{m-1}(x) + \eta \cdot h_m(x)$$

gdzie $h_m$ jest dopasowane do **pseudoreszt** (ujemnego gradientu funkcji straty). Popularne implementacje: **XGBoost**, **LightGBM**, **CatBoost**.

In [ ]:
# Random Forest
rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf.fit(X_train, y_train)

print(f"Train accuracy: {rf.score(X_train, y_train):.3f}")
print(f"Test accuracy: {rf.score(X_test, y_test):.3f}")

# Feature importance (uśredniona po wielu drzewach → stabilniejsza)
importance_rf = rf.feature_importances_
plt.figure(figsize=(8, 4))
plt.barh(features, importance_rf, color="forestgreen")
plt.xlabel("Importance")
plt.title("Ważność cech — Random Forest")
plt.show()

## 3. SVM (Support Vector Machine)

**Idea:** Znajdź **hiperpłaszczyznę**, która najlepiej rozdziela klasy, maksymalizując **margines** — odległość do najbliższych punktów (wektorów nośnych).

### Problem optymalizacji — Hard Margin SVM

Dla danych liniowo separowalnych chcemy znaleźć hiperpłaszczyznę $\mathbf{w} \cdot \mathbf{x} + b = 0$ taką, że:

$$\min_{\mathbf{w}, b} \frac{1}{2}\|\mathbf{w}\|^2$$

$$\text{s.t.} \quad y_i(\mathbf{w} \cdot \mathbf{x}_i + b) \geq 1 \quad \forall i$$

Margines = $\frac{2}{\|\mathbf{w}\|}$, więc minimalizacja $\|\mathbf{w}\|$ = **maksymalizacja marginesu**.

### Soft Margin SVM (C-SVM)

Dla danych nieseparowalnych liniowo — dopuszczamy naruszenia marginesu za pomocą zmiennych swobodnych $\xi_i \geq 0$:

$$\min_{\mathbf{w}, b, \xi} \frac{1}{2}\|\mathbf{w}\|^2 + C \sum_{i=1}^{n} \xi_i$$

$$\text{s.t.} \quad y_i(\mathbf{w} \cdot \mathbf{x}_i + b) \geq 1 - \xi_i, \quad \xi_i \geq 0$$

- **Duży C** → mało naruszeń (mały margines, ryzyko overfittingu)
- **Mały C** → więcej naruszeń (duży margines, bardziej regularny model)

### Wektory nośne (Support Vectors)

Wektory nośne to punkty, dla których $y_i(\mathbf{w} \cdot \mathbf{x}_i + b) = 1$ (leżą na granicy marginesu). Tylko one wpływają na pozycję hiperpłaszczyzny — reszta danych można usunąć bez zmiany modelu.

### Problem dualny i Kernel Trick

Problem dualny (Lagrange'a):

$$\max_{\alpha} \sum_{i=1}^{n} \alpha_i - \frac{1}{2}\sum_{i,j} \alpha_i \alpha_j y_i y_j (\mathbf{x}_i \cdot \mathbf{x}_j)$$

$$\text{s.t.} \quad 0 \leq \alpha_i \leq C, \quad \sum_i \alpha_i y_i = 0$$

**Kluczowe spostrzeżenie:** dane pojawiają się tylko jako iloczyny skalarne $\mathbf{x}_i \cdot \mathbf{x}_j$. Możemy zastąpić je **funkcją jądra** $K(\mathbf{x}_i, \mathbf{x}_j)$:

$$K(\mathbf{x}_i, \mathbf{x}_j) = \phi(\mathbf{x}_i) \cdot \phi(\mathbf{x}_j)$$

Obliczamy iloczyn skalarny w przestrzeni wyższego wymiaru **bez jawnego** wyznaczania $\phi$!

### Popularne kernele

| Kernel | Wzór | Hiperparametry |
|--------|------|---------------|
| **Linear** | $K(\mathbf{x}, \mathbf{z}) = \mathbf{x} \cdot \mathbf{z}$ | brak |
| **Polynomial** | $K(\mathbf{x}, \mathbf{z}) = (\gamma \mathbf{x} \cdot \mathbf{z} + r)^d$ | $d$ (stopień), $\gamma$, $r$ |
| **RBF (Gaussian)** | $K(\mathbf{x}, \mathbf{z}) = \exp(-\gamma\|\mathbf{x}-\mathbf{z}\|^2)$ | $\gamma$ |
| **Sigmoid** | $K(\mathbf{x}, \mathbf{z}) = \tanh(\gamma \mathbf{x} \cdot \mathbf{z} + r)$ | $\gamma$, $r$ |

### Wpływ gamma w RBF
- **Mały $\gamma$** → gładka granica (duży "zasięg" każdego punktu), ryzyko underfittingu
- **Duży $\gamma$** → skomplikowana granica (mały "zasięg"), ryzyko overfittingu

### SVM — dlaczego standaryzacja jest konieczna?

SVM opiera się na odległościach. Jeśli cechy mają różne skale, cecha o dużej skali zdominuje obliczenie $\|\mathbf{w}\|$ i iloczynu skalarnego.

In [ ]:
# SVM — WYMAGA standaryzacji!
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

svm = SVC(kernel="rbf", C=1.0, gamma="scale", random_state=42)
svm.fit(X_train_s, y_train)

print(f"Train accuracy: {svm.score(X_train_s, y_train):.3f}")
print(f"Test accuracy: {svm.score(X_test_s, y_test):.3f}")

## 4. Grid Search — szukanie najlepszych hiperparametrów

### Przeszukiwanie siatki (Grid Search)

Próbujemy **wszystkie kombinacje** hiperparametrów i wybieramy najlepszą wg cross-validation.

Złożoność: jeśli mamy $k$ hiperparametrów z odpowiednio $n_1, n_2, \ldots, n_k$ wartościami i $F$-fold CV:

$$\text{Liczba fitów} = n_1 \times n_2 \times \ldots \times n_k \times F$$

### Random Search — alternatywa

Zamiast sprawdzać **wszystkie** kombinacje, losujemy $N$ konfiguracji. Bergstra & Bengio (2012) pokazali, że Random Search jest efektywniejszy niż Grid Search, gdy tylko kilka hiperparametrów ma znaczący wpływ.

### Bayesian Optimization

Zaawansowana metoda — buduje **model zastępczy** (surrogate, np. Gaussian Process) funkcji celu i wybiera kolejne konfiguracje wg kryterium **Expected Improvement (EI)**:

$$\text{EI}(x) = \mathbb{E}[\max(f(x) - f^*, 0)]$$

Biblioteki: **Optuna**, **Hyperopt**, **scikit-optimize**.

In [ ]:
# Grid Search dla SVM
param_grid = {
 "C": [0.1, 1, 10, 100],
 "gamma": ["scale", "auto", 0.01, 0.1],
 "kernel": ["rbf", "linear"]
}

grid = GridSearchCV(
 SVC(random_state=42),
 param_grid,
 cv=5, # 5-fold cross-validation
 scoring="accuracy",
 verbose=0
)
grid.fit(X_train_s, y_train)

print(f"Najlepsze parametry: {grid.best_params_}")
print(f"Najlepszy CV score: {grid.best_score_:.3f}")
print(f"Test accuracy: {grid.score(X_test_s, y_test):.3f}")

## 5. Porównanie algorytmów

In [ ]:
# Porównanie 4 algorytmów na Wine dataset
wine = load_wine()
X_w, y_w = wine.data, wine.target

scaler_w = StandardScaler()
X_w_scaled = scaler_w.fit_transform(X_w)

models = {
 "KNN (k=5)": KNeighborsClassifier(n_neighbors=5),
 "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
 "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
 "SVM (RBF)": SVC(kernel="rbf", random_state=42),
}

results = {}
for name, model in models.items():
 scores = cross_val_score(model, X_w_scaled, y_w, cv=5, scoring="accuracy")
 results[name] = scores
 print(f"{name:20s}: {scores.mean():.3f} ± {scores.std():.3f}")

# Boxplot porównawczy
plt.figure(figsize=(10, 5))
plt.boxplot(results.values(), labels=results.keys())
plt.ylabel("Accuracy")
plt.title("Porównanie algorytmów (5-fold CV, Wine dataset)")
plt.xticks(rotation=15)
plt.grid(True, alpha=0.3)
plt.show()

---
---
## Zadania do samodzielnego rozwiązania

---

### Zadanie 1: Drzewo decyzyjne — wpływ max_depth (łatwe)

Trenuj drzewa decyzyjne z `max_depth` od 1 do 20 na zbiorze Iris.
Narysuj wykres train accuracy vs test accuracy w zależności od głębokości.
Wyznacz optymalną głębokość.

In [ ]:
# Zadanie 1
# TWÓJ KOD TUTAJ

### Zadanie 2: Random Forest — ile drzew? (średnie)

Zbadaj, jak liczba drzew (`n_estimators`: 1, 5, 10, 20, 50, 100, 200, 500) wpływa na accuracy.
Użyj cross-validation. Narysuj wykres z error barami.

In [ ]:
# Zadanie 2
# TWÓJ KOD TUTAJ

### Zadanie 3: Pełny pipeline ML (trudne)

Przeprowadź kompletny pipeline ML na zbiorze Wine:
1. Podział na train/test (80/20)
2. Standaryzacja (fit na train!)
3. Grid Search na Random Forest:
 - `n_estimators`: [50, 100, 200]
 - `max_depth`: [3, 5, 10, None]
 - `min_samples_split`: [2, 5, 10]
4. Wypisz najlepsze parametry i test accuracy
5. Narysuj confusion matrix i classification report
6. Narysuj feature importance (top 5)

In [ ]:
# Zadanie 3: Pełny pipeline
# TWÓJ KOD TUTAJ

### Zadanie 4: Granice decyzyjne 2D (trudne)

Wygeneruj 2D dane (`make_classification`, 2 cechy) i narysuj granice decyzyjne dla:
- KNN (k=3)
- Decision Tree (max_depth=5)
- SVM (RBF)

3 subploty obok siebie. Użyj `plt.contourf` do kolorowania regionów decyzyjnych.

**Wskazówka:** Stwórz gęstą siatkę (`np.meshgrid`) i przewidź klasę dla każdego punktu siatki.

In [ ]:
# Zadanie 4: Granice decyzyjne
np.random.seed(42)
# TWÓJ KOD TUTAJ

### Zadanie 5: Feature importance z Random Forest (srednie)

1. Zaladuj zbior Wine (`sklearn.datasets.load_wine`)
2. Wytrenuj RandomForestClassifier
3. Wyciagnij `feature_importances_` i narysuj wykres slupkowy (posortowany)
4. Wytrenuj model ponownie uzywajac tylko **5 najwazniejszych cech** — jak zmienia sie accuracy?
5. Uzyj takze `permutation_importance` — czy ranking sie zgadza?

In [ ]:
# Zadanie 5: Feature importance
from sklearn.datasets import load_wine
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import numpy as np
import matplotlib.pyplot as plt
# TWOJ KOD TUTAJ

### Zadanie 6: Ensemble stacking (trudne)

Zaimplementuj **prostego stacka**:

**Level 0 (base models):**
- DecisionTree
- KNN
- SVM (rbf)

**Level 1 (meta-learner):**
- LogisticRegression na predykcjach z level 0

Uzyj cross-validation na level 0 (zeby uniknac data leakage).
Porownaj accuracy stacka z kazdym modelem osobno na zbiorze testowym.

In [ ]:
# Zadanie 6: Stacking ensemble
from sklearn.datasets import load_wine
from sklearn.model_selection import cross_val_predict, train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
import numpy as np
# TWOJ KOD TUTAJ

### Zadanie 7: Bagging vs Boosting
Na zbiorze danych `breast_cancer` (sklearn) porównaj:
- `BaggingClassifier` z bazowym drzewem decyzyjnym
- `GradientBoostingClassifier`
- `RandomForestClassifier`

Użyj 5-fold CV. Wyświetl mean +/- std accuracy i czas treningu.
Narysuj learning curves dla wszystkich trzech modeli.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import BaggingClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
import time

# TWOJ KOD TUTAJ


### Zadanie 8: Feature Importance i selekcja cech
Na zbiorze `wine` (sklearn) wytrenuj Random Forest.
1. Wyświetl feature importances (MDI) jako wykres słupkowy, posortowany malejąco
2. Wytrenuj model tylko na top-5 cechach — porównaj accuracy z modelem na wszystkich
3. Użyj `permutation_importance` — czy ranking cech jest taki sam jak z MDI?

In [ ]:
from sklearn.datasets import load_wine
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt
import numpy as np

# TWOJ KOD TUTAJ


---

## Dodatek OAI: Ćwiczenia w stylu olimpiadowym

### Kontekst olimpiadowy
Klasyczne ML jest kluczowe na OAI! Przykłady:
- **II OAI, Etap 1:** Wykrywanie halucynacji LLM — rozwiązanie z **XGBoost** dało najlepsze wyniki (ROC AUC)
- **II OAI, Etap 1:** Zaburzenia EKG — klasyfikacja bez GPU, czyste ML (balanced accuracy)
- **III OAI, Etap 1:** Zmiany semantyczne — ensemble HistGradientBoosting + LogisticRegression dał 100/100 pkt!
- **II OAI, Etap 2:** Kredytobranie — XAI/counterfactual explanations dla klasyfikatora

### Ćwiczenie OAI-6A: Ensemble w stylu olimpiadowym (zmiany semantyczne)

Wzorzec z modelowego rozwiązania III OAI:
1. Stwórz cechy (feature engineering) z surowych danych
2. Wytrenuj HistGradientBoosting i LogisticRegression
3. Połącz w stacking ensemble
4. Optymalizuj próg decyzyjny pod balanced accuracy

### Ćwiczenie OAI-6B: XGBoost z ograniczeniami olimpiadowymi

Wytrenuj XGBoost na syntetycznym zbiorze (5000 próbek, 20 cech). 
Wymogi: trening < 10 sekund, raportuj ROC AUC.

### Ćwiczenie OAI-6C: Counterfactual Explanations (XAI)

Wytrenuj prosty klasyfikator (np. RandomForest) na zbiorze Iris.
Dla wybranej próbki znajdź **minimalną zmianę cech** potrzebną do zmiany predykcji.
To jest uproszczona wersja zadania "Kredytobranie" z II OAI, etap 2.

In [ ]:
import numpy as np
import time
from sklearn.datasets import make_classification, load_iris
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.ensemble import (RandomForestClassifier, 
 HistGradientBoostingClassifier,
 StackingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, roc_auc_score

# === OAI-6A: Stacking Ensemble (wzorzec III OAI) ===
print("=== Stacking Ensemble (wzorzec III OAI — zmiany semantyczne) ===")
X, y = make_classification(n_samples=3000, n_features=15, n_classes=2,
 n_informative=10, random_state=42)

# Modele bazowe (jak w modelowym rozwiązaniu OAI)
estimators = [
 ('hgb', HistGradientBoostingClassifier(max_iter=100, random_state=42)),
 ('lr', LogisticRegression(max_iter=1000, random_state=42))
]

# Meta-klasyfikator (stacking)
stacking = StackingClassifier(
 estimators=estimators,
 final_estimator=LogisticRegression(),
 cv=5
)

start = time.time()
scores = cross_val_score(stacking, X, y, cv=5, scoring='balanced_accuracy')
elapsed = time.time() - start
print(f" Stacking bal_acc = {scores.mean():.4f} ± {scores.std():.4f} (czas: {elapsed:.1f}s)")

# === OAI-6B: XGBoost z limitem czasowym ===
print("\n=== XGBoost z limitem 10s ===")
try:
 from xgboost import XGBClassifier

 xgb = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, 
 use_label_encoder=False, eval_metric='logloss',
 random_state=42)
 start = time.time()
 scores_xgb = cross_val_score(xgb, X, y, cv=5, scoring='roc_auc')
 elapsed = time.time() - start

 print(f" XGBoost ROC AUC = {scores_xgb.mean():.4f} ± {scores_xgb.std():.4f} (czas: {elapsed:.1f}s)")
 print(f" {' W limicie' if elapsed < 10 else ' Przekroczono limit!'}")
except ImportError:
 print(" XGBoost nie zainstalowany. Zainstaluj: pip install xgboost")

# === OAI-6C: Prosta wersja Counterfactual Explanations ===
print("\n=== Counterfactual Explanation (wzorzec: Kredytobranie, II OAI etap 2) ===")
iris = load_iris()
X_iris, y_iris = iris.data, iris.target

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_iris, y_iris)

# Wybierz próbkę do wyjaśnienia
sample_idx = 0
original = X_iris[sample_idx].copy()
original_pred = clf.predict([original])[0]
print(f" Oryginalna próbka: klasa = {iris.target_names[original_pred]}")
print(f" Cechy: {dict(zip(iris.feature_names, original))}")

# Szukaj minimalnej zmiany (brute force — prosty counterfactual)
best_change = None
best_distance = float('inf')
for feat_idx in range(4):
 for delta in np.arange(-3, 3.1, 0.1):
 modified = original.copy()
 modified[feat_idx] += delta
 new_pred = clf.predict([modified])[0]
 if new_pred != original_pred:
 dist = abs(delta)
 if dist < best_distance:
 best_distance = dist
 best_change = (iris.feature_names[feat_idx], delta, iris.target_names[new_pred])

if best_change:
 print(f" Counterfactual: zmień '{best_change[0]}' o {best_change[1]:+.1f} → klasa '{best_change[2]}'")
 print(f" Dystans L1: {best_distance:.1f}")

print("\n Na OAI Kredytobranie trzeba było znaleźć OPTYMALNE counterfactuals minimalizujące dystans!")